# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** Refresh / Content Opportunity Scoring.

**Research question:** given limited editorial review capacity, which pages should be reviewed
first for refresh, and why?

**Unit of analysis:** one content page, described by its trailing 90-day search and engagement
signals.

**Decision supported:** which pages make it into a content team's review queue this sprint.

**Action:** an editor pulls the top-ranked pages and reviews/refreshes them; low-priority pages
are monitored, not touched.

**Cost of a wrong call:** a false positive wastes editor time on a page that wasn't really at
risk; a false negative lets a genuinely declining, high-demand page go unreviewed for another
cycle. Editor time is the scarce resource, so precision at the top of the queue matters more than
covering every page.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
print("Lane: Refresh / Content Opportunity Scoring")
print("Question: which pages should be reviewed first, given limited editor capacity?")


Lane: Refresh / Content Opportunity Scoring
Question: which pages should be reviewed first, given limited editor capacity?


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source:** the small anonymized starter dataset shipped in this repo
(`data/raw/content_refresh_anonymized.csv`), 30,000 pseudonymized content items across 32
pseudonymized clients, each described by 44 trailing-90-day search/engagement columns. (The full
79M-row Hugging Face warehouse release was also available this track; this capstone's model and
baseline are built on the starter slice, consistent with every earlier weekly notebook, so the
Results comparison stays apples-to-apples with w04–w07.)

**Eligibility filter (applied throughout, matching every earlier week):** `impressions_90d > 0`
and `content_age_days >= 90` — 30,000 of 30,000 rows pass this filter in the starter slice.

**Excluded, deliberately:**
- No raw query, URL, title, or client-identifying text — none of it ships in this dataset, and
  none is reconstructed here.
- No FlyRank product decision flags (`health_score`, `priority_score`, `action_type`) — the
  starter data ships observable signals only, by design.
- `trend_pct` and the 30d/prev-30d breakdown fields are used only to *verify* the label, never as
  model features (see Methodology).


In [2]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

print(f"total rows: {len(df):,}")
print(f"eligible rows (impressions_90d>0 & age>=90d): {len(eligible):,} ({len(eligible)/len(df):.1%})")
print(f"clients: {eligible['client_id'].nunique()}")
print(f"label positive rate (proxy): {eligible['is_declining_label'].mean():.1%}")


total rows: 30,000
eligible rows (impressions_90d>0 & age>=90d): 30,000 (100.0%)
clients: 32
label positive rate (proxy): 54.2%


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label (proxy, not a verified future outcome):** `is_declining_label = trend_direction == "down"`
— a same-window classification, not a prediction of the future. Flagged as a limitation since
Week 2 and carried through every week since.

**Baseline (w04):** a transparent rule — CTR gap vs. position-tier benchmark (volume-filtered to
avoid a low-impression outlier bug caught during Week 4) × demand weight, with a small capped
staleness bonus. One reason code: `ctr_underperform_vs_position`.

**Model (w05, re-validated w06):** Gradient Boosting Classifier, chosen after comparing Logistic
Regression (AUC 0.528) and Random Forest (AUC 0.599) on the same features and split — Gradient
Boosting had the clearest edge (AUC 0.614).

**Features (10, all knowable before the decision point):** `impressions_90d`, `clicks_90d`,
`sessions_90d`, `avg_position`, `ctr`, `content_age_days`, `days_since_last_update`, `word_count`,
`engagement_rate`, `scroll_rate`.

**Validation design:** `GroupShuffleSplit` by `client_id` (75/25), verified zero client overlap.
Week 6 demonstrated why this matters directly: the same model under a plain random split scored
Precision@50 = 0.94 with 31 of 32 clients leaking across train/test — the honest, client-held-out
number is 0.76.

**Leakage audit (w03, w06):** the honest/leaky trap in Week 3 showed a feature-window-only AUC of
0.595 jump to a meaningless 1.000 when the literal target-window value was smuggled in as a
feature. The final 10-feature set was re-audited in Week 6: no feature is one of the fields
`trend_direction` is directly computed from, and no FlyRank product flag was used as a feature.


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

features = ["impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
            "content_age_days", "days_since_last_update", "word_count",
            "engagement_rate", "scroll_rate"]

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

# honest, client-grouped split (the number this paper reports)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(eligible, groups=eligible["client_id"]))
train, test = eligible.iloc[train_idx].copy(), eligible.iloc[test_idx].copy()
overlap = set(train["client_id"]) & set(test["client_id"])
print(f"client overlap (grouped split): {len(overlap)} (must be 0)")

# rebuild the w04 baseline rule identically
tier_mean_ctr = eligible[eligible["impressions_90d"] >= 100].groupby("position_tier", observed=True)["ctr"].mean()
eligible["tier_mean_ctr"] = eligible["position_tier"].map(tier_mean_ctr)
eligible["ctr_gap"] = (eligible["tier_mean_ctr"] - eligible["ctr"]).clip(lower=0)
eligible["demand_weight"] = np.log1p(eligible["impressions_90d"])
eligible["staleness_bucket"] = pd.cut(eligible["days_since_last_update"], bins=[-1, 90, 180, 365, 100000],
                                        labels=["<90d", "90-180d", "180-365d", "365d+"])
staleness_component = ((eligible["staleness_bucket"] == "90-180d").astype(float)) * \
    (eligible["ctr_gap"] * eligible["demand_weight"]).median() * 0.15
eligible["baseline_action_score"] = eligible["ctr_gap"] * eligible["demand_weight"] + staleness_component
test["baseline_action_score"] = eligible.loc[test.index, "baseline_action_score"]

# fit the model
X_train, y_train = train[features].fillna(0), train["is_declining_label"]
X_test, y_test = test[features].fillna(0), test["is_declining_label"]
model = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
model.fit(X_train, y_train)
test["model_probability"] = model.predict_proba(X_test)[:, 1]

print("methodology pipeline rebuilt: baseline + model, honest split, ready for Results.")


client overlap (grouped split): 0 (must be 0)


methodology pipeline rebuilt: baseline + model, honest split, ready for Results.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*


In [4]:
baseline_p50 = precision_at_k(test["baseline_action_score"].values, y_test.values, k=50)
model_p50 = precision_at_k(test["model_probability"].values, y_test.values, k=50)
baseline_auc = roc_auc_score(y_test, test["baseline_action_score"])
model_auc = roc_auc_score(y_test, test["model_probability"])

results_table = pd.DataFrame({
    "method": ["Baseline (CTR-gap rule)", "Gradient Boosting model"],
    "precision_at_50": [round(baseline_p50, 3), round(model_p50, 3)],
    "roc_auc": [round(baseline_auc, 3), round(model_auc, 3)],
})
print(results_table.to_string(index=False))
print(f"\nmodel lift over baseline: {model_p50 - baseline_p50:+.3f} Precision@50 "
      f"({(model_p50/baseline_p50 - 1):+.0%})")

# context: how this compares to a random-split (leaky) version, for honesty
X_all, y_all = eligible[features].fillna(0), eligible["is_declining_label"]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_all, y_all, test_size=0.25, random_state=42, stratify=y_all)
model_leaky = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
model_leaky.fit(X_tr2, y_tr2)
leaky_p50 = precision_at_k(model_leaky.predict_proba(X_te2)[:, 1], y_te2.values, k=50)
print(f"\n(context, not reported as the real number) same model, random row split: "
      f"Precision@50 = {leaky_p50:.3f} -- inflated by client leakage, see Limitations.")


                 method  precision_at_50  roc_auc
Baseline (CTR-gap rule)             0.62    0.586
Gradient Boosting model             0.76    0.614

model lift over baseline: +0.140 Precision@50 (+23%)



(context, not reported as the real number) same model, random row split: Precision@50 = 0.940 -- inflated by client leakage, see Limitations.


## 5. Limitations

*What this work cannot claim.*

- **Proxy label, not a verified outcome.** `trend_direction == "down"` is computed from the same
  window as the features. This paper reports what the model predicts about a *current* same-window
  label, not a genuine future outcome — a stricter prior-90-days-predicts-next-30-days label (as
  piloted in w03, AUC 0.595 on a mid-panel month) is the honest next step, not what's reported here.
- **Small, single-snapshot dataset.** 30,000 pages, 32 clients, one time snapshot. Not tested on
  a different vertical, a different time period, or clients outside this dataset.
- **Feature importance skews toward volume/age, not content quality.** `impressions_90d` and
  `content_age_days` account for 57% of the model's importance combined; CTR contributes ~5%,
  engagement rate ~1%. A "high risk" score is partly just "this page is old and/or busy," not a
  diagnosis of *why* a page struggles.
- **No causal claim.** Nothing here shows that refreshing a page *causes* recovery — that would
  require a controlled experiment, not this observational data.
- **No claim about Google's ranking algorithm.** Every signal here is an observed outcome
  (impressions, clicks, position), never the mechanism behind it.
- **Validation, while honest, is still bounded.** The client-grouped split (Week 6) closed the
  most dramatic leak found (0.94 → 0.76), but a leakage audit can only rule out what it checks for
  — it is not proof of the absence of every possible leak.


In [5]:
print("Limitations are qualitative by design -- no code output needed beyond the numbers already")
print("established above (feature importances, split comparison). See markdown cell above.")


Limitations are qualitative by design -- no code output needed beyond the numbers already
established above (feature importances, split comparison). See markdown cell above.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Built in w07: every eligible page is scored by the validated model and mapped to one of five
archetypes, each with an action and a reason code. `refresh_priority` and `ctr_opportunity` pages
are the ones an editor should look at first; `protect` pages should be left alone and distributed,
not edited; `monitor_routine` needs no action this cycle.


In [6]:
eligible["decline_probability"] = model.predict_proba(eligible[features].fillna(0))[:, 1]

def assign_archetype(row):
    if row["decline_probability"] >= 0.6 and row["impressions_90d"] >= 500:
        return "refresh_priority", "refresh_priority_review", "high_risk_high_demand"
    if row["ctr"] < 0.5 * row["tier_mean_ctr"] and row["impressions_90d"] >= 500:
        return "ctr_opportunity", "review_for_ctr_fix", "ctr_underperform_vs_position"
    if row["decline_probability"] >= 0.6 and row["impressions_90d"] < 500:
        return "monitor_low_demand", "monitor_low_priority", "high_risk_low_demand"
    if row["position_tier"] in ("top_3", "page_1") and row["decline_probability"] < 0.3:
        return "protect", "protect_distribute", "strong_stable_asset"
    return "monitor_routine", "monitor_routine", ""

results = eligible.apply(assign_archetype, axis=1, result_type="expand")
eligible["archetype"], eligible["action"], eligible["reason_code"] = results[0], results[1], results[2]

archetype_counts = eligible["archetype"].value_counts()
print("Ranked action playbook -- archetype mix across all eligible pages:")
print(archetype_counts)
print(f"\nactionable share (needs review this cycle): "
      f"{eligible['archetype'].isin(['refresh_priority','ctr_opportunity','monitor_low_demand']).mean():.1%}")


Ranked action playbook -- archetype mix across all eligible pages:
archetype
monitor_routine       10239
refresh_priority       9043
monitor_low_demand     5715
protect                2676
ctr_opportunity        2327
Name: count, dtype: int64

actionable share (needs review this cycle): 57.0%


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Two figures, already generated and committed in Week 7 (`work/figures/`):
- `archetype_mix.png` — the recommendation breakdown above, as a chart.
- `decline_by_freshness.png` — the decay/refresh insight (decline probability by freshness tier).

Both are referenced directly by the deployed paper's Results and Recommendations sections.


In [7]:
import os
print("Figures already committed in work/figures/:")
for f in os.listdir("work/figures"):
    print(" -", f)

print("\nMetric receipts already committed in work/outputs/:")
for f in sorted(os.listdir("work/outputs")):
    if f.endswith(".json"):
        print(" -", f)


Figures already committed in work/figures/:
 - archetype_mix.png
 - decline_by_freshness.png

Metric receipts already committed in work/outputs/:
 - w04_baseline_summary.json
 - w06_validation_summary.json
 - w07_playbook_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.